# Quest Hero: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_hero")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from quest_hero import *
from quest_hero.game_world import GameWorld, CellType, NPC, NPC_CATALOG
from quest_hero.hero import Hero
from quest_hero.agent import TOOLS_DESCRIPTION, parse_tool_call
from quest_hero.oracle import ORACLE_TEMPLATE, llm_oracle
from quest_hero.display import render_grid
print("Quest Hero loaded!")

In [ ]:
# Show the full map
hero, world, tools = create_game()
print(render_grid(world, hero, show_all=True))

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: systematic exploration with priority-based actions.
- Pick up items on current cell
- Talk to NPCs/innkeepers/shopkeepers
- Buy/craft when we have the materials
- Fight dragons when we have the right weapon
- Move toward unvisited cells, preferring cells with interesting content

In [ ]:
from collections import deque

def _bfs_next_step(start, target, world):
    """BFS pathfinding that navigates around mountains."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def think_rule_based(hero: Hero, world: GameWorld, history: list[dict]) -> str:
    row, col = hero.position
    cell = world.get_cell(row, col)

    # --- Priority 1: Pick up items on current cell ---
    if cell.items:
        return 'TOOL: pick_up()'

    # --- Priority 2: Interact with current cell ---
    if cell.cell_type == CellType.NPC and cell.npc:
        npc_name = cell.npc.name
        talked = sum(1 for e in hero.journal if npc_name in e)
        if talked == 0:
            return 'TOOL: talk(question="What can you tell me? What quests or dangers should I know about?")'
        if cell.npc_id == "traveling_merchant" and not hero.has_item("Letter") and not world.errand_letter_picked_up:
            return 'TOOL: talk(question="Do you have any letter or delivery job for me?")'
        if cell.npc_id == "forest_witch" and hero.has_item("Ghostcaps"):
            return 'TOOL: talk(question="I have Ghostcaps. Can you brew medicine?")'
        if cell.npc_id == "old_hermit" and hero.has_item("Letter"):
            return 'TOOL: talk(question="I have a letter for you from the merchant.")'

    if cell.cell_type == CellType.INN:
        if hero.position == (2, 5):
            if hero.has_item("Herbal Tea"):
                return 'TOOL: talk(question="I brought herbal tea from your brother!")'
            if hero.has_item("Medicine"):
                return 'TOOL: talk(question="I have medicine for your wife!")'
        if hero.position == (7, 4) and not world.errand_tea_picked_up:
            return 'TOOL: talk(question="Hello, do you need any help?")'
        if hero.hearts < hero.MAX_HEARTS and hero.gold >= 2:
            return 'TOOL: rest()'

    if cell.cell_type == CellType.SHOP:
        if hero.has_item("Ember Ore") and hero.gold >= 1 and hero.position == (6, 2):
            return 'TOOL: buy(item="Sunblade")'
        if hero.has_item("Moonstone") and hero.gold >= 1 and hero.position == (4, 1):
            return 'TOOL: buy(item="Moonstone Lantern")'
        if hero.has_item("Dragon Scales") and hero.position == (4, 1):
            return 'TOOL: buy(item="sell Dragon Scales")'

    if cell.cell_type == CellType.DRAGON:
        if cell.dragon_name == "Frost Wyrm" and hero.has_item("Sunblade") and world.frost_wyrm_alive:
            return 'TOOL: fight()'
        if cell.dragon_name == "Shadow Beast" and hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
            return 'TOOL: fight()'

    if hero.hearts <= 1:
        if hero.has_item("Health Potion"):
            return 'TOOL: use(item="Health Potion")'
        if hero.has_item("Bread"):
            return 'TOOL: use(item="Bread")'

    # --- Priority 3: Navigate to targets using BFS ---
    targets = []

    if hero.has_item("Letter"):
        targets.append((7, 1))
    if hero.has_item("Herbal Tea"):
        targets.append((2, 5))
    if hero.has_item("Medicine"):
        targets.append((2, 5))
    if hero.has_item("Ember Ore"):
        targets.append((6, 2))
    if hero.has_item("Sunblade") and world.frost_wyrm_alive:
        targets.append((2, 1))
    if hero.has_item("Moonstone"):
        targets.append((4, 1))
    if hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
        targets.append((4, 6))
    if hero.has_item("Dragon Scales"):
        targets.append((4, 1))

    for tc in [(5, 0), (6, 6), (0, 7)]:
        if tc not in hero.visited:
            targets.append(tc)

    if (7, 1) not in hero.visited:
        targets.append((7, 1))
    if (7, 4) not in hero.visited:
        targets.append((7, 4))
    if (5, 3) not in hero.visited:
        targets.append((5, 3))
    if (3, 4) not in hero.visited:
        targets.append((3, 4))

    if not hero.has_item("Ember Ore") and not hero.has_item("Sunblade"):
        if (2, 0) not in hero.visited:
            targets.append((2, 0))

    seen = set()
    unique = []
    for t in targets:
        if t not in seen and t != (row, col):
            seen.add(t)
            unique.append(t)

    for target in unique:
        d = _bfs_next_step((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if (nr, nc) not in hero.visited and world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    return 'TOOL: look()'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

---
## Solution: TODO 2 — LLM Think Function

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

In [ ]:
def think_llm(hero: Hero, world: GameWorld, history: list[dict]) -> str:
    system = f"""You are an AI agent controlling a hero in an RPG grid world.

YOUR GOAL: Collect 10 gold and survive (keep at least 1 heart).

AVAILABLE TOOLS:
{TOOLS_DESCRIPTION}

CRITICAL RULES:
- Respond with exactly ONE tool call in the format: TOOL: tool_name(arg="value")
- NEVER call look(). You automatically see your surroundings every turn. Calling look() wastes a turn.
- NEVER call status(). Your status is shown to you every turn already.
- If pick_up() returns "nothing to pick up", the cell is EMPTY. Leave immediately and never try pick_up there again.
- If talk() returns "no one to talk to", you are not on an NPC/inn/shop cell. Move to one first.

EXPLORATION IS KEY:
- You MUST explore the full 8x8 map. There are NPCs, shops, treasures, and quests scattered everywhere.
- ALWAYS prioritize moving to UNVISITED cells. If you have been at the same position for 2+ turns, MOVE somewhere new.
- Never go back to a cell you already looted or fully explored.

GOLD STRATEGY (you need 10):
- Treasures give only 1 gold each (3 on the map = 3 gold max). This is NOT enough.
- You MUST talk to every NPC you find — they give quests worth 2 gold each.
- You MUST collect materials, craft weapons at shops, and fight dragons (3 gold each).
- Talk to innkeepers too — they have delivery errands.

QUEST CHAINS:
- NPCs give hints about where to find materials and how to craft weapons.
- Shops can craft weapons from materials you find (costs gold + materials).
- Fight dragons ONLY with the correct weapon, or you lose a heart.

Think step by step, then output EXACTLY one TOOL: line."""

    # Build user message with current context
    recent = history[-15:]
    history_text = "\n".join(
        f"  [{h['role']}] {h['content'][:200]}" for h in recent
    )

    journal_text = "\n".join(
        f"  - {entry[:150]}" for entry in hero.journal[-10:]
    ) if hero.journal else "  (no journal entries yet)"

    row, col = hero.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label

    # Track visited cells for the LLM
    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(hero.visited))

    # Stuck detection: count consecutive turns at same position
    stuck_warning = ""
    same_pos_count = 0
    for h in reversed(history):
        if h["role"] == "observation" and f"position ({row}, {col})" in h["content"]:
            same_pos_count += 1
        elif h["role"] == "observation":
            break
    if same_pos_count >= 3:
        stuck_warning = f"\n⚠️ WARNING: You have been at position ({row}, {col}) for {same_pos_count} turns! You are STUCK. Move to an unvisited cell NOW.\n"

    user_msg = f"""CURRENT STATUS:
{hero.status_text()}

CURRENT CELL: {cell_desc} (type: {cell.cell_type.value})
{stuck_warning}
VISITED CELLS ({len(hero.visited)} of 64): {visited_str}
UNVISITED NEIGHBORS: check the directions below and prefer cells you haven't been to.

RECENT HISTORY:
{history_text}

JOURNAL (NPC conversations and key events):
{journal_text}

What should I do next? Remember: explore aggressively, talk to every NPC, complete quests. Output one TOOL: call."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_msg,
        config=genai.types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=500,
        ),
    )

    return response.text

In [ ]:
oracle_fn = lambda npc, q, h: llm_oracle(npc, q, h, client)

result = play_with_llm(
    think_fn=lambda hero, world, history: think_llm(hero, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")